# GENOVISTA 2.0 — Google Colab launcher
Run each cell in order. The public link is temporary and stays available only while this runtime is connected.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
print("Uploaded:", zip_name)

In [ ]:
import os, zipfile, shutil

project_dir = "/content/genovista_project"
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)
os.makedirs(project_dir, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(project_dir)
app_dir = os.path.join(project_dir, "GENOVISTA_REBUILT")
if not os.path.isdir(app_dir):
    # ZIP may have the project files at its root.
    app_dir = project_dir
assert os.path.isfile(os.path.join(app_dir, "app.py")), "app.py was not found in the uploaded ZIP"
os.chdir(app_dir)
print("Project folder:", app_dir)
print("app.py found:", os.path.isfile("app.py"))

In [ ]:
%pip install -r requirements.txt

In [ ]:
import subprocess, time, os

# Start Streamlit in the background and retain its process handle.
log_handle = open("/content/genovista_streamlit.log", "w")
streamlit_process = subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py",
     "--server.address", "0.0.0.0", "--server.port", "8501",
     "--server.headless", "true"],
    cwd=app_dir, stdout=log_handle, stderr=subprocess.STDOUT
)
time.sleep(8)
print("Streamlit running:", streamlit_process.poll() is None)
if streamlit_process.poll() is not None:
    print(open("/content/genovista_streamlit.log").read())

In [ ]:
# Install Cloudflare Tunnel and create a temporary public HTTPS URL.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, time
tunnel_log_handle = open("/content/cloudflared.log", "w")
tunnel_process = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=tunnel_log_handle, stderr=subprocess.STDOUT
)
public_url = None
for _ in range(40):
    time.sleep(2)
    log_text = open("/content/cloudflared.log", errors="replace").read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
print("GENOVISTA public URL:", public_url or "Not detected; inspect /content/cloudflared.log")

In [ ]:
# Optional diagnostics: view the Streamlit log if the page does not load.
print(open("/content/genovista_streamlit.log", errors="replace").read()[-5000:])